In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from google.colab import files
from io import BytesIO
from PIL import Image

model = keras.applications.VGG16()

uploaded = files.upload()

img = Image.open(BytesIO(uploaded['Panda.jpg']))
plt.imshow(img)

filename = next(iter(uploaded))
img = Image.open(BytesIO(uploaded[filename])).convert('RGB')   # 3 kanal təmin edir
img = img.resize((224, 224))                                    # ölçünü dəyiş

# приводим к входному формату VGG-сети
img = np.array(img)
x = keras.applications.vgg16.preprocess_input(img)
print(x.shape)
x = np.expand_dims(x, axis=0)
print(x.shape)

# прогоняем через сеть
res = model.predict(x)
print(np.argmax(res))

from tensorflow.keras.applications.vgg16 import decode_predictions
# ən yüksək 5 proqnoz
preds = decode_predictions(res, top=5)[0]
for class_id, name, prob in preds:
    print(f"{name:25s} {prob*100:.2f}%  ({class_id})")

## Izah

In [ ]:
import numpy as np
# NumPy — Python-da ədədlər, massivlər və çoxölçülü array-lərlə
# işləmək üçün istifadə olunan əsas kitabxanadır.
#
# np — NumPy üçün verdiyimiz qısa addır.
#
# Bu proqramda NumPy-dan:
# - şəkli massivə çevirmək üçün (np.array)
# - şəkilə "batch" ölçüsü əlavə etmək üçün (np.expand_dims)
# - ən yüksək ehtimallı sinfin indeksini tapmaq üçün (np.argmax)
#   istifadə edəcəyik.


import matplotlib.pyplot as plt
# matplotlib.pyplot — Python-da qrafik, şəkil və digər vizual
# elementləri göstərmək üçün istifadə olunan kitabxanadır.
#
# plt — matplotlib.pyplot üçün verdiyimiz qısa addır.
#
# Bu kodda plt.imshow() ilə yüklənmiş şəkli ekranda göstərəcəyik.
# (MNIST koduna nisbətən fərq: orada plt import olunub, amma
# istifadə olunmurdu. Burada isə istifadə olunur.)


from tensorflow import keras
# TensorFlow daxilindəki Keras API-ni import edirik.
#
# Bu dəfə keras vasitəsilə:
# - hazır VGG16 modelini yükləyəcəyik (keras.applications.VGG16)
# - VGG16-ya uyğun preprocess_input funksiyasından istifadə edəcəyik
#   (keras.applications.vgg16.preprocess_input)
#
# Əvvəlki kodda modeli özümüz layer-layer qururduq (keras.Sequential).
# İndi isə modeli qurmuruq və öyrətmirik — hazır, artıq öyrədilmiş
# modeldən istifadə edirik.


from google.colab import files
# google.colab — yalnız Google Colab mühitində işləyən kitabxanadır.
#
# files modulu kompüterdən Colab-a fayl yükləmək (upload)
# və ya Colab-dan kompüterə fayl endirmək (download) üçündür.
#
# DİQQƏT: Bu sətr Colab-dan kənarda (məsələn, öz kompüterindəki
# Jupyter və ya VS Code-da) işləməyəcək, ModuleNotFoundError verəcək.


from io import BytesIO
# io — Python-un standart giriş/çıxış kitabxanasıdır.
#
# BytesIO — yaddaşda (RAM-da) saxlanılan baytları sanki həqiqi fayl
# kimi oxumağa imkan verən sinifdir.
#
# Niyə lazımdır?
# files.upload() faylı diskə yox, yaddaşa "bayt" (bytes) şəklində qaytarır.
# Image.open() isə fayl kimi bir şey gözləyir.
# BytesIO bu baytları "saxta fayl"a çevirərək problemi həll edir.


from PIL import Image
# PIL (Pillow) — Python-da şəkillərlə işləmək üçün kitabxanadır.
#
# Image sinfi ilə:
# - şəkli açmaq            (Image.open)
# - rəng rejimini dəyişmək (.convert)
# - ölçüsünü dəyişmək      (.resize)
# mümkündür.


model = keras.applications.VGG16()
# Hazır VGG16 modelini yükləyirik.
#
# VGG16 — Oxford Universitetində hazırlanmış məşhur CNN modelidir.
# Adındakı 16 — çəkiləri olan 16 layer (13 Conv2D + 3 Dense) deməkdir.
#
# Heç bir parametr verməsək, defolt olaraq:
#
# weights='imagenet'
# → model ImageNet datasetində ARTIQ öyrədilmiş çəkilərlə gəlir.
# → ImageNet: 1,2 milyondan çox şəkil, 1000 fərqli sinif
#   (pişik, it, panda, maşın, stul və s.).
#
# include_top=True
# → modelin sonundakı Dense qatları da daxildir,
#   yəni model birbaşa 1000 sinif üzrə proqnoz verir.
#
# Giriş forması: (224, 224, 3)
# → 224 hündürlük, 224 en, 3 kanal (RGB rəngli şəkil).
#
# Çıxış forması: (1000,)
# → 1000 sinfin hər biri üçün ehtimal.
#
# Model təxminən 138 milyon parametrə malikdir.
#
# DİQQƏT:
# Bu sətr ilk dəfə işləyəndə çəkiləri internetdən yükləyir
# (təxminən 528 MB). Ona görə ilk işə düşmə bir az çəkə bilər.
#
# Nəzərə al: bu modeli öyrətmək lazım deyil.
# Ona görə burada nə compile(), nə də fit() var.


uploaded = files.upload()
# Colab-da "Choose Files" düyməsi çıxarır və şəkil yükləməyə imkan verir.
#
# Nəticə uploaded dəyişəninə lüğət (dictionary) şəklində yazılır:
#
# {
#   'Panda.jpg': b'\xff\xd8\xff\xe0...'
# }
#
# Burada:
# açar (key)     → faylın adı
# qiymət (value) → faylın məzmunu (baytlar)
#
# Bu sətr işləyəndə proqram dayanıb fayl seçməyini gözləyir.


img = Image.open(BytesIO(uploaded['Panda.jpg']))
# Yüklənmiş şəkli açır.
#
# uploaded['Panda.jpg'] → lüğətdən həmin faylın baytlarını götürür.
# BytesIO(...)          → baytları "fayl kimi" oxunan obyektə çevirir.
# Image.open(...)       → PIL ilə şəkli açır.
#
# DİQQƏT:
# Fayl adı kodda əllə yazılıb: 'Panda.jpg'.
# Yüklədiyin faylın adı bundan fərqlidirsə (məsələn, 'Panda (4).jpg'),
# KeyError xətası alacaqsan.
#
# Bu xətanı aşağıdakı filename = next(iter(uploaded)) sətri də
# həll edə bilər — o, adı avtomatik götürür. Yəni bu sətr əslində
# artıqdır, sadəcə şəkli göstərmək üçün istifadə olunur.


plt.imshow(img)
# Şəkli ekranda göstərir.
#
# Bunun məqsədi yüklədiyimiz şəklin düzgün olduğunu gözlə yoxlamaqdır.
#
# Bu mərhələdə şəkil hələ ilkin ölçüsündədir (məsələn, 1200 × 800),
# 224 × 224-ə kiçildilməyib.


filename = next(iter(uploaded))
# Yüklənmiş faylın adını avtomatik götürür.
#
# uploaded lüğətdir. iter(uploaded) onun açarlarını (fayl adlarını)
# gəzən iterator yaradır. next(...) həmin iteratordan ilk elementi alır.
#
# Yəni bu sətr "yüklənən ilk faylın adını ver" deməkdir.
#
# Məsələn uploaded = {'Panda.jpg': b'...'} olarsa:
#
# filename = 'Panda.jpg'
#
# Faydası: fayl adını əllə yazmaq lazım qalmır.


img = Image.open(BytesIO(uploaded[filename])).convert('RGB')   # 3 kanal təmin edir
# Şəkli yenidən açırıq, amma bu dəfə adı filename dəyişənindən alırıq
# və dərhal .convert('RGB') tətbiq edirik.
#
# .convert('RGB') nə üçün lazımdır?
#
# Şəkillər fərqli rəng rejimlərində ola bilər:
#
# 'RGB'  → 3 kanal (qırmızı, yaşıl, mavi)
# 'RGBA' → 4 kanal (RGB + şəffaflıq). PNG şəkillərdə tez-tez olur.
# 'L'    → 1 kanal (qara-ağ)
# 'P'    → palitralı rejim
#
# VGG16 isə dəqiq 3 kanal gözləyir.
# Şəkil RGBA (4 kanal) və ya qara-ağ (1 kanal) olarsa,
# model forma xətası verərdi.
#
# .convert('RGB') hər halda şəkli 3 kanallı formaya gətirir.


img = img.resize((224, 224))                                    # ölçünü dəyiş
# Şəklin ölçüsünü 224 × 224 piksel edir.
#
# Niyə məhz 224?
# → VGG16 məhz (224, 224, 3) ölçülü şəkillərlə öyrədilib.
# → Başqa ölçü verilsə, model xəta verəcək.
#
# DİQQƏT: PIL-də resize((en, hündürlük)) yazılır. Burada ikisi də
# 224-dür, ona görə fərq etmir.
#
# Qeyd:
# Bu sadə resize şəkli "sıxa" və ya "uza" bilər. Məsələn, 1200 × 800
# ölçülü şəkil 224 × 224-ə gətiriləndə nisbətlər pozulur (panda
# bir az enli görünə bilər). Adətən bu, nəticəyə çox təsir etmir.


# приводим к входному формату VGG-сети
# Yuxarıdakı rus dilindəki şərh: "VGG şəbəkəsinin giriş formatına gətiririk".
img = np.array(img)
# PIL şəklini NumPy massivinə çevirir.
#
# Bundan əvvəl img PIL obyekti idi. İndi isə adi ədədlər massividir.
#
# Formanın dəyişməsi:
#
# PIL şəkli  →  massiv (224, 224, 3)
#
# Burada:
#
# 224 → hündürlük
# 224 → en
# 3   → kanal sayı (R, G, B)
#
# Hər piksel 0 ilə 255 arasında tam ədəddir (dtype: uint8).


x = keras.applications.vgg16.preprocess_input(img)
# Şəkli VGG16-nın gözlədiyi formata salır.
#
# DİQQƏT — bu MNIST koduna nisbətən əsas fərqlərdən biridir!
# MNIST-də şəkli sadəcə 255-ə bölürdük (0–1 aralığı).
# VGG16 isə BELƏ ÖYRƏDİLMƏYİB. Onun öz xüsusi ön emalı var.
# Ona görə 255-ə bölmək burada YANLIŞ olardı.
#
# preprocess_input() iki şey edir:
#
# 1. RGB → BGR
#    Kanalların sırasını dəyişir: (R, G, B) → (B, G, R).
#    Çünki VGG16 orijinal olaraq belə öyrədilib.
#
# 2. Hər kanaldan ImageNet-in orta qiymətini çıxır
#    (təxminən B: 103.94, G: 116.78, R: 123.68).
#    Bu, məlumatı sıfır ətrafında mərkəzləşdirir.
#    Nəticədə piksel qiymətləri artıq 0–255 yox, təxminən
#    -124 ilə +151 arasında olur və ədədlər kəsirli (float32) olur.
#
# Forma dəyişmir: (224, 224, 3) olaraq qalır.
#
# Hər şəbəkənin öz preprocess_input funksiyası var
# (resnet50.preprocess_input, mobilenet.preprocess_input və s.).
# Həmişə modelə uyğun olanı seçmək lazımdır.


print(x.shape)
# x massivinin formasını çap edir.
#
# Nəticə belə olacaq:
#
# (224, 224, 3)
#
# Hələ ki bu TƏK şəkildir, batch ölçüsü yoxdur.


x = np.expand_dims(x, axis=0)
# Massivin əvvəlinə yeni ölçü əlavə edir.
#
# Niyə lazımdır?
# Keras modelləri həmişə şəkillərin DƏSTƏSİNİ (batch) gözləyir:
#
# (şəkil sayı, hündürlük, en, kanal)
#
# Bizdə isə yalnız 1 şəkil var və forması (224, 224, 3)-dür.
# Bir şəkli də "1 şəkillik dəstə" kimi təqdim etməliyik.
#
# axis=0 → yeni ölçü ən əvvəldə (birinci mövqedə) əlavə olunur.
#
# Nəticədə:
#
# (224, 224, 3)  →  (1, 224, 224, 3)
#
# Buradakı 1 → dəstədə 1 şəkil olduğunu bildirir.
#
# MNIST koduyla müqayisə:
# orada expand_dims(..., axis=3) ilə ƏN SONA kanal ölçüsü əlavə edirdik.
# Burada isə ƏN ƏVVƏLƏ batch ölçüsü əlavə edirik, ona görə axis=0 yazılır.


print(x.shape)
# Yenə formanı çap edir.
#
# Nəticə belə olacaq:
#
# (1, 224, 224, 3)
#
# İndi şəkil modelə verilməyə hazırdır.


# прогоняем через сеть
# Yuxarıdakı rus dilindəki şərh: "şəbəkədən keçiririk".
res = model.predict(x)
# Modeldən proqnoz alır.
#
# predict() şəkli şəbəkədən keçirir və hər sinif üçün ehtimal qaytarır.
#
# Nəticə res dəyişəninə yazılır. Onun forması:
#
# (1, 1000)
#
# Burada:
#
# 1    → bir şəkil verdik
# 1000 → ImageNet-in 1000 sinfinin hər biri üçün ehtimal
#
# Son qat softmax olduğu üçün 1000 ehtimalın cəmi təxminən 1-dir.
#
# Məsələn:
#
# res[0][388] ≈ 0.99   (panda sinfi)
# res[0][0]   ≈ 0.00   (başqa sinif)
#
# Qeyd:
# predict() işləyəndə konsolda "1/1 [====]" kimi irəliləyiş
# çubuğu çıxır. Bu normaldır.


print(np.argmax(res))
# Ən böyük ehtimala malik sinfin İNDEKSİNİ çap edir.
#
# np.argmax() massivdəki ən böyük qiymətin yerini (indeksini) qaytarır.
#
# Məsələn:
#
# [0.01, 0.02, 0.95, 0.02]  →  argmax = 2
#
# Bizim halda nəticə 388 oldu.
# Bu rəqəm sinfin ADI deyil, ImageNet siyahısındakı sıra nömrəsidir.
# 388 → "giant panda" sinfinə uyğundur.
#
# Problem: 388 rəqəmindən sinfin adını özümüz bilmirik.
# Bunu aşağıda decode_predictions ilə həll edirik.


from tensorflow.keras.applications.vgg16 import decode_predictions
# decode_predictions funksiyasını import edirik.
#
# Bu funksiya modelin verdiyi 1000 ədədlik ehtimal massivini
# insan üçün oxunaqlı formaya çevirir: sinfin adı + ehtimal.
#
# Sinif adlarını özü imagenet_class_index.json faylından oxuyur.
# Bu faylı ilk işləmədə avtomatik yükləyir, buna görə
# internet bağlantısı lazımdır.
#
# Beləliklə, sinif adlarını GitHub-dan əl ilə axtarmağa ehtiyac qalmır.


# ən yüksək 5 proqnoz
preds = decode_predictions(res, top=5)[0]
# Ən yüksək ehtimallı 5 sinfi tapır.
#
# decode_predictions(res, top=5):
# → res massivindən ən böyük 5 ehtimalı seçir
# → hər biri üçün (sinif_id, ad, ehtimal) üçlüyü qaytarır
#
# top=5 → ən yaxşı 5 nəticəni göstər deməkdir.
# İstəsən top=1 (yalnız ən yaxşı) və ya top=10 yaza bilərsən.
#
# Nəticənin quruluşu:
#
# [
#   [ ('n02510455', 'giant_panda', 0.99),
#     ('n02509815', 'lesser_panda', 0.00),
#     ... ]
# ]
#
# Yəni siyahının içində siyahı var: hər şəkil üçün ayrıca bir siyahı.
# Biz yalnız 1 şəkil verdiyimiz üçün xarici siyahıda 1 element var.
#
# [0] → həmin ilk (və yeganə) şəklin nəticə siyahısını götürür.
#
# Üçlüyün hissələri:
#
# sinif_id → ImageNet-də sinfin unikal kodu (məsələn, n02510455)
# ad       → sinfin adı (məsələn, giant_panda)
# ehtimal  → 0 ilə 1 arasında ədəd (məsələn, 0.99)


for class_id, name, prob in preds:
    # 5 nəticənin hər birini ardıcıl gəzir.
    #
    # Hər dövrdə üçlük üç dəyişənə ayrılır:
    #
    # class_id → sinif kodu
    # name     → sinif adı
    # prob     → ehtimal (0–1 arası)
    #
    # Siyahı ehtimala görə azalan sıradadır,
    # yəni birinci sətir ən güclü proqnozdur.

    print(f"{name:25s} {prob*100:.2f}%  ({class_id})")
    # Hər nəticəni səliqəli formada çap edir.
    #
    # f"..." → f-string-dir, içində {} ilə dəyişən yazmağa imkan verir.
    #
    # {name:25s}
    # → name-i mətn (s = string) kimi, 25 simvol enində yazır.
    # → Qısa adların sağını boşluqla doldurur ki, sütunlar düz düzülsün.
    #
    # {prob*100:.2f}
    # → ehtimalı 100-ə vurur (0.99 → 99) yəni faizə çevirir.
    # → .2f → vergüldən sonra 2 rəqəm göstərir (99.47).
    #
    # %
    # → sadəcə faiz işarəsidir, mətn kimi çap olunur.
    #
    # ({class_id})
    # → sonda sinif kodunu mötərizədə göstərir.
    #
    # Nəticə təxminən belə görünəcək:
    #
    # giant_panda               99.xx%  (n02510455)
    # ...
    #
    # Qeyd: başqa 4 sinfin ehtimalı adətən çox kiçik olur
    # (0.xx% və ya daha az), çünki model panda ilə bağlı
    # yüksək əminliyə malikdir.